# Vision Assist V2 — Voice-Enabled Detection Demo

**Version:** V2 — object detection with spoken alerts  
**Model:** Existing trained YOLOv8n checkpoint (no retraining)  
**Purpose:** Run inference on a small set of ROD validation images, generate spoken alerts, and export a short MP4 demo.

## Run order
Run cells from top to bottom. Attach the ROD dataset to Kaggle before running.

**Important:** This notebook demonstrates detection and speech on sample images. It is not a live webcam system or a safety-validated navigation aid.

## 1. Environment and paths
Install the inference dependency, check GPU availability, and define output paths.

In [ ]:
%pip install -q ultralytics==8.4.175

from pathlib import Path
import hashlib
import importlib.util
import shutil
import subprocess
import urllib.request
import json
import cv2
import torch
from IPython.display import Audio, Video, Image as NotebookImage, display
from ultralytics import YOLO

PROJECT_DIR = Path("/kaggle/working/vision_assist")
OUTPUT_DIR = PROJECT_DIR / "v2_voice_demo"
MODEL_PATH = PROJECT_DIR / "rod_yolov8n_50ep" / "weights" / "best.pt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Output directory:", OUTPUT_DIR)

## 2. Restore the existing trained checkpoint
Download the model from the project GitHub repository only if it is not already present. This cell does not train the model.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_URL = (
    "https://raw.githubusercontent.com/"
    "SudhaMsr/VisionAssist-MVP/main/"
    "models/rod_yolov8n_50ep_best.pt"
)

if not MODEL_PATH.exists():
    print("Downloading existing checkpoint...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

assert MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 1_000_000, (
    "Checkpoint is missing or unexpectedly small. Check the GitHub model URL."
)

sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
model = YOLO(str(MODEL_PATH))

print("Model loaded:", MODEL_PATH.name)
print("Checkpoint size (MB):", round(MODEL_PATH.stat().st_size / 1e6, 2))
print("SHA-256:", sha256)
print("Number of classes:", len(model.names))

## 3. Find validation images
The dataset is used only as demo input here. The notebook does not train or evaluate the model.

In [ ]:
DATASET_ROOT = Path(
    "/kaggle/input/datasets/abtinzandi/"
    "obstacle-detection-dataset/ROD-Dataset/dataset"
)
VALID_IMAGES = DATASET_ROOT / "valid" / "images"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert VALID_IMAGES.is_dir(), (
    f"Validation images not found at {VALID_IMAGES}. "
    "In Kaggle, use Add Input to attach the ROD dataset."
)

image_files = sorted(
    p for p in VALID_IMAGES.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
)
assert image_files, f"No images found in {VALID_IMAGES}"

# Keep the demo short and repeatable.
sample_images = image_files[:8]
print(f"Using {len(sample_images)} validation images.")
for path in sample_images:
    print(path.name)

## 4. Install offline speech support
Install eSpeak NG so alerts can be generated without an online text-to-speech service.

In [ ]:
speech_engine = shutil.which("espeak-ng") or shutil.which("espeak")

if speech_engine is None:
    update = subprocess.run(
        ["apt-get", "update", "-qq"],
        capture_output=True, text=True
    )
    install = subprocess.run(
        ["apt-get", "install", "-y", "-qq", "espeak-ng"],
        capture_output=True, text=True
    )
    speech_engine = shutil.which("espeak-ng") or shutil.which("espeak")
    if speech_engine is None:
        raise RuntimeError(
            "Could not install eSpeak NG. "
            + install.stderr[-1000:]
        )

print("Speech engine:", speech_engine)

## 5. Detect objects and create spoken alerts
For each sample image, run inference, save an annotated frame, and create a concise spoken summary of the detected classes.

In [ ]:
CONFIDENCE_THRESHOLD = 0.35
IMAGE_SIZE = 640

spoken_alerts = []
saved_frames = []
demo_rows = []

for index, image_path in enumerate(sample_images, start=1):
    result = model.predict(
        source=str(image_path),
        imgsz=IMAGE_SIZE,
        conf=CONFIDENCE_THRESHOLD,
        verbose=False
    )[0]

    frame_path = OUTPUT_DIR / f"frame_{index:02d}.jpg"
    if not cv2.imwrite(str(frame_path), result.plot()):
        raise RuntimeError(f"Could not save annotated frame: {frame_path}")
    saved_frames.append(frame_path)

    detected_names = []
    if result.boxes is not None and len(result.boxes) > 0:
        detected_names = sorted({
            model.names[int(class_id)]
            for class_id in result.boxes.cls.tolist()
        })

    if detected_names:
        alert = "Caution. " + ", ".join(detected_names[:3]) + " detected."
    else:
        alert = "No objects detected."

    spoken_alerts.append(alert)
    demo_rows.append({
        "image": image_path.name,
        "detected_classes": ", ".join(detected_names),
        "spoken_alert": alert
    })
    print(f"{index}/{len(sample_images)} | {image_path.name} | {alert}")

transcript_path = OUTPUT_DIR / "spoken_alerts.txt"
transcript_path.write_text("\n".join(spoken_alerts), encoding="utf-8")

metadata_path = OUTPUT_DIR / "demo_results.json"
metadata_path.write_text(
    json.dumps({
        "version": "V2",
        "model": MODEL_PATH.name,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "image_count": len(sample_images),
        "results": demo_rows
    }, indent=2),
    encoding="utf-8"
)

display(NotebookImage(filename=str(saved_frames[0])))

## 6. Generate audio with a pause between alerts
Each image gets its own spoken segment with a short pause so the warnings remain distinct.

In [ ]:
import wave

audio_segments = []
audio_params = None

for index, alert in enumerate(spoken_alerts, start=1):
    segment_path = OUTPUT_DIR / f"alert_{index:02d}.wav"
    subprocess.run(
        [speech_engine, "-v", "en", "-s", "145", "-w", str(segment_path), alert],
        check=True,
        capture_output=True,
        text=True
    )

    with wave.open(str(segment_path), "rb") as wav_in:
        params = wav_in.getparams()
        audio_data = wav_in.readframes(wav_in.getnframes())

    if audio_params is None:
        audio_params = params
    elif (
        params.nchannels != audio_params.nchannels
        or params.sampwidth != audio_params.sampwidth
        or params.framerate != audio_params.framerate
        or params.comptype != audio_params.comptype
    ):
        raise RuntimeError("Speech segments have incompatible audio formats.")

    silence_duration_seconds = 0.6
    silence_bytes = (
        b"\x00" * params.sampwidth * params.nchannels
        * int(params.framerate * silence_duration_seconds)
    )
    audio_segments.append(audio_data + silence_bytes)

AUDIO_PATH = OUTPUT_DIR / "vision_assist_v2_voice.wav"
with wave.open(str(AUDIO_PATH), "wb") as wav_out:
    wav_out.setparams(audio_params)
    for segment in audio_segments:
        wav_out.writeframes(segment)

print("Audio saved:", AUDIO_PATH)
print("Alerts:")
for alert in spoken_alerts:
    print("-", alert)

display(Audio(filename=str(AUDIO_PATH)))

## 7. Export the annotated video with speech
Each annotated image is shown for two seconds. The audio contains the matching spoken alerts.

In [ ]:
FFMPEG = shutil.which("ffmpeg")
assert FFMPEG, "ffmpeg is not available in this Kaggle session."

SILENT_VIDEO = OUTPUT_DIR / "vision_assist_v2_silent.mp4"
VIDEO_PATH = OUTPUT_DIR / "vision_assist_v2_voice_demo.mp4"

first_frame = cv2.imread(str(saved_frames[0]))
height, width = first_frame.shape[:2]
fps = 2
seconds_per_frame = 2

writer = cv2.VideoWriter(
    str(SILENT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

for frame_path in saved_frames:
    frame = cv2.imread(str(frame_path))
    if frame is None:
        raise RuntimeError(f"Could not read frame: {frame_path}")
    if frame.shape[:2] != (height, width):
        frame = cv2.resize(frame, (width, height))
    for _ in range(fps * seconds_per_frame):
        writer.write(frame)
writer.release()

subprocess.run([
    FFMPEG, "-y",
    "-i", str(SILENT_VIDEO),
    "-i", str(AUDIO_PATH),
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    str(VIDEO_PATH)
], check=True, capture_output=True, text=True)

print("V2 video created:", VIDEO_PATH)
print("Video size (MB):", round(VIDEO_PATH.stat().st_size / 1e6, 2))
display(Video(str(VIDEO_PATH), embed=True, width=720))

## 8. Outputs and limitations

Main outputs are saved under `/kaggle/working/vision_assist/v2_voice_demo/`:

- `vision_assist_v2_voice_demo.mp4` — annotated video with speech
- `vision_assist_v2_voice.wav` — spoken warnings
- `spoken_alerts.txt` — readable alert transcript
- `demo_results.json` — per-image output summary
- `frame_*.jpg` — annotated images

### Scope
- Uses the existing trained checkpoint; no retraining occurs.
- Demonstrates image-based inference and spoken alerts.
- It is **not** a live webcam feed.
- Class detections are not proof that an object is a hazard.
- Do not use this experimental prototype as a safety-critical navigation aid.

### GitHub note
Commit this notebook and the lightweight text metadata/transcript if useful. Avoid committing generated videos, repeated WAV files, image frames, dataset images, or secrets by default; store demo media as a release asset or Git LFS if you intentionally want them versioned.